# Exercise 3.1: Mainz OSM SQL Planner with an LLM and DuckDB

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yfeng-hsm/KI_Geodatenanalyse_SS26/blob/main/lectures/03_llm_basics/notebooks/exercise_3_1_llm_osm_tool_widget_mainz.ipynb)

This notebook uses a small local OpenStreetMap snapshot for Mainz city center and queries it with DuckDB. The LLM task is to translate a natural-language question into SQL over a fixed table schema.

The workflow is deliberately split into visible steps:

1. Load a local Mainz OSM POI CSV into DuckDB.
2. Run a simple hand-written SQL query first, without any LLM.
3. Ask the Uni Mainz KI-Chat@JGU model for a JSON plan containing SQL.
4. Display the raw model response and the exact SQL returned by the model.
5. Validate the SQL before execution.
6. Execute the SQL in DuckDB.
7. Visualize returned rows on a Folium map.

This avoids live Overpass API instability during the main teaching exercise.

## Learning Outcomes

After this exercise you should be able to:

- Load a small OSM-derived dataset into DuckDB.
- Run SQL queries against geospatial point attributes.
- Prompt an LLM to generate SQL for a fixed table schema.
- Inspect and validate model-generated SQL before execution.
- Execute validated SQL and visualize rows with longitude/latitude columns.
- Explain why a local fixture can be better than a live API for teaching and testing.

## 1. Colab Setup

In [ ]:
!pip -q install duckdb pandas folium ipywidgets requests


## 2. Imports and Data Download

The CSV fixture was created from a small OpenStreetMap API extract for Mainz city center.

Source bbox: west `8.254`, south `49.996`, east `8.282`, north `50.005`. This extent includes Mainz Hauptbahnhof and the central Altstadt area.

License note: OpenStreetMap data is © OpenStreetMap contributors and distributed under the ODbL 1.0.

In [ ]:
from __future__ import annotations

import getpass
import json
import re
from pathlib import Path
from typing import Any

import duckdb
import folium
import pandas as pd
import requests
from IPython.display import Markdown, display
from folium.plugins import MarkerCluster

DATA_URL = "https://raw.githubusercontent.com/yfeng-hsm/KI_Geodatenanalyse_SS26/main/lectures/03_llm_basics/data/mainz_city_center_osm_pois.csv"
METADATA_URL = "https://raw.githubusercontent.com/yfeng-hsm/KI_Geodatenanalyse_SS26/main/lectures/03_llm_basics/data/mainz_city_center_osm_pois_metadata.json"
DATA_FILE = Path("mainz_city_center_osm_pois.csv")
METADATA_FILE = Path("mainz_city_center_osm_pois_metadata.json")
EXPECTED_BBOX = [8.254, 49.996, 8.282, 50.005]


def download_fixture() -> None:
    data_response = requests.get(DATA_URL, timeout=60)
    data_response.raise_for_status()
    DATA_FILE.write_bytes(data_response.content)

    metadata_response = requests.get(METADATA_URL, timeout=60)
    metadata_response.raise_for_status()
    METADATA_FILE.write_bytes(metadata_response.content)


needs_download = not DATA_FILE.exists() or not METADATA_FILE.exists()
if not needs_download:
    existing_metadata = json.loads(METADATA_FILE.read_text(encoding="utf-8"))
    needs_download = existing_metadata.get("bbox_west_south_east_north") != EXPECTED_BBOX

if needs_download:
    download_fixture()

metadata = json.loads(METADATA_FILE.read_text(encoding="utf-8"))
DATA_BBOX = metadata["bbox_west_south_east_north"]  # west, south, east, north
DATA_BOUNDS = [[DATA_BBOX[1], DATA_BBOX[0]], [DATA_BBOX[3], DATA_BBOX[2]]]  # Folium: [[south, west], [north, east]]
DATA_CENTER = ((DATA_BBOX[1] + DATA_BBOX[3]) / 2, (DATA_BBOX[0] + DATA_BBOX[2]) / 2)

print("Downloaded/loaded:", DATA_FILE)
print("Rows in fixture metadata:", metadata.get("rows"))
print("BBox:", metadata.get("bbox_west_south_east_north"))
print("Map bounds:", DATA_BOUNDS)
print("Tag counts:")
for key, value in metadata.get("tag_key_counts", {}).items():
    print(f"- {key}: {value}")


## 3. Load the CSV into DuckDB

The notebook loads DuckDB Spatial before creating tables. The main table is `osm_pois`. A second small table, `known_places`, stores named Mainz reference points inside the fixture extent that SQL can use for projected metric distance queries.

In [ ]:
con = duckdb.connect(database=":memory:")

# DuckDB spatial is required for ST_Point, ST_Transform, ST_DWithin, and ST_Distance.
# In Colab this downloads the extension once for the current runtime.
con.execute("INSTALL spatial;")
con.execute("LOAD spatial;")

csv_path = str(DATA_FILE).replace("'", "''")
con.execute(f"""
CREATE OR REPLACE VIEW osm_pois AS
SELECT *
FROM read_csv_auto('{csv_path}', all_varchar=true)
""")

KNOWN_PLACES = {
    "mainz_hbf": {"label": "Mainz Hauptbahnhof", "lat": 50.0010, "lon": 8.2587},
    "mainz_dom": {"label": "Mainz Cathedral", "lat": 49.9995, "lon": 8.2742},
    "mainz_theater": {"label": "Staatstheater Mainz", "lat": 50.0002, "lon": 8.2714},
}

known_places_df = pd.DataFrame([
    {"place_id": place_id, **place}
    for place_id, place in KNOWN_PLACES.items()
])
con.register("known_places", known_places_df)

print(con.execute("SELECT COUNT(*) AS n FROM osm_pois").fetchdf())
print(con.execute("SELECT * FROM known_places").fetchdf())
print("DuckDB spatial extension loaded.")


## 4. DuckDB Smoke Test Without an LLM

Always test the local data path first. This query should work before asking the model to generate SQL.

In [ ]:
smoke_sql = """
SELECT
  name,
  amenity,
  CAST(lon AS DOUBLE) AS lon,
  CAST(lat AS DOUBLE) AS lat
FROM osm_pois
WHERE amenity = 'cafe'
ORDER BY name
LIMIT 10;
"""

print(smoke_sql)
smoke_df = con.execute(smoke_sql).fetchdf()
display(smoke_df)
assert len(smoke_df) > 0, "Smoke test should return cafes from the local Mainz fixture."
print("DuckDB smoke test passed.")


## 5. A Hand-Written Distance Query

This query finds cafes within 800 m of Mainz Hauptbahnhof. It creates WGS84 point geometries, transforms them to `EPSG:25832` for metric analysis around Mainz, then uses `ST_DWithin` and `ST_Distance`.

In [ ]:
handwritten_distance_sql = """
WITH center_projected AS (
  SELECT
    place_id,
    label,
    ST_Transform(ST_Point(lon, lat), 'EPSG:4326', 'EPSG:25832', true) AS geom_25832
  FROM known_places
  WHERE place_id = 'mainz_hbf'
), pois_projected AS (
  SELECT
    p.name,
    p.amenity,
    CAST(p.lon AS DOUBLE) AS lon,
    CAST(p.lat AS DOUBLE) AS lat,
    ST_Transform(
      ST_Point(CAST(p.lon AS DOUBLE), CAST(p.lat AS DOUBLE)),
      'EPSG:4326',
      'EPSG:25832',
      true
    ) AS geom_25832
  FROM osm_pois AS p
  WHERE p.amenity = 'cafe'
), ranked AS (
  SELECT
    p.name,
    p.amenity,
    p.lon,
    p.lat,
    ST_Distance(p.geom_25832, c.geom_25832) AS distance_m
  FROM pois_projected AS p
  CROSS JOIN center_projected AS c
  WHERE ST_DWithin(p.geom_25832, c.geom_25832, 800)
)
SELECT name, amenity, lon, lat, round(distance_m, 1) AS distance_m
FROM ranked
ORDER BY distance_m
LIMIT 20;
"""

print(handwritten_distance_sql)
handwritten_df = con.execute(handwritten_distance_sql).fetchdf()
display(handwritten_df)
assert len(handwritten_df) > 0, "Expected at least one cafe near Mainz Hbf."


## 6. Map Any SQL Result with `lon` and `lat` Columns

In [ ]:
def add_fixture_bbox_layer(m: folium.Map) -> folium.Map:
    """Add the local OSM fixture coverage as a toggleable map layer."""
    bbox_group = folium.FeatureGroup(name="OSM fixture bbox", show=True)
    folium.Rectangle(
        bounds=DATA_BOUNDS,
        color="#f97316",
        weight=3,
        fill=True,
        fill_color="#f97316",
        fill_opacity=0.08,
        tooltip="Local Mainz OSM fixture extent",
        popup=(
            f"OSM fixture bbox<br>"
            f"west={DATA_BBOX[0]}, south={DATA_BBOX[1]}<br>"
            f"east={DATA_BBOX[2]}, north={DATA_BBOX[3]}"
        ),
    ).add_to(bbox_group)
    bbox_group.add_to(m)
    return m


def map_sql_result(df: pd.DataFrame, title: str = "Mainz SQL result") -> folium.Map | None:
    if df.empty:
        display(Markdown("No rows to map."))
        return None
    if "lon" not in df.columns or "lat" not in df.columns:
        display(Markdown("The result has no `lon` and `lat` columns, so it cannot be mapped."))
        return None

    m = folium.Map(location=DATA_CENTER, zoom_start=14, tiles="OpenStreetMap", control_scale=True)
    add_fixture_bbox_layer(m)
    html = f'<div style="position: fixed; top: 10px; left: 50px; z-index: 9999; background: white; padding: 8px 10px; border: 1px solid #999; font-size: 14px;"><strong>{title}</strong></div>'
    m.get_root().html.add_child(folium.Element(html))
    cluster = MarkerCluster(name="SQL result").add_to(m)

    for _, row in df.iterrows():
        lon = float(row["lon"])
        lat = float(row["lat"])
        popup_parts = []
        for col in ["name", "amenity", "tourism", "shop", "tag_key", "tag_value", "distance_m"]:
            if col in row and pd.notna(row[col]) and str(row[col]) != "":
                popup_parts.append(f"{col}: {row[col]}")
        popup = "<br>".join(popup_parts) if popup_parts else "OSM POI"
        folium.CircleMarker((lat, lon), radius=5, color="blue", fill=True, fill_opacity=0.75, popup=popup).add_to(cluster)

    m.fit_bounds(DATA_BOUNDS)
    folium.LayerControl().add_to(m)
    display(m)
    return m


map_sql_result(handwritten_df, "Cafes within 800 m of Mainz Hbf")


## 7. Uni Mainz KI-Chat API

Now that the local DuckDB path works, ask the LLM to generate SQL.

Create an API key in the KI-Chat@JGU web interface and paste it below. Do not store API keys in notebook cells.

In [ ]:
KI_CHAT_API_BASE_URL = "https://ki-chat.uni-mainz.de/api"
KI_CHAT_API_KEY = getpass.getpass("Paste your KI-Chat@JGU API key for this session: ").strip()
if not KI_CHAT_API_KEY:
    raise RuntimeError("No API key entered.")

ki_chat_headers = {
    "Authorization": f"Bearer {KI_CHAT_API_KEY}",
    "Content-Type": "application/json",
}
print("KI-Chat API key loaded for this notebook session.")


In [ ]:
def ki_chat_request(method: str, endpoint: str, **kwargs: Any) -> Any:
    url = f"{KI_CHAT_API_BASE_URL}{endpoint}"
    response = requests.request(method, url, headers=ki_chat_headers, timeout=90, **kwargs)
    if not response.ok:
        print(f"HTTP {response.status_code} for {method} {endpoint}")
        try:
            print(json.dumps(response.json(), indent=2, ensure_ascii=False))
        except ValueError:
            print(response.text[:1000])
        response.raise_for_status()
    return response.json()


def preview_json(data: Any, max_chars: int = 4000) -> None:
    text = json.dumps(data, indent=2, ensure_ascii=False)
    print(text[:max_chars] + ("\n..." if len(text) > max_chars else ""))


In [ ]:
models_response = ki_chat_request("GET", "/models")
model_ids = [m.get("id") for m in models_response.get("data", []) if m.get("id")]
print("Available model IDs:")
for model_id in model_ids:
    print("-", model_id)

PREFERRED_CHAT_MODELS = ["Qwen3 235B", "Qwen3 235B VL", "GPT OSS 120B"]
chat_model = next((model for model in PREFERRED_CHAT_MODELS if model in model_ids), None)
if chat_model is None:
    chat_model = model_ids[0] if model_ids else PREFERRED_CHAT_MODELS[0]
print(f"\nUsing chat model: {chat_model}")


## 8. SQL Generation Prompt

The model must return JSON with one SQL query. The SQL is displayed before it is executed.

In [ ]:
SQL_PLANNER_SYSTEM_PROMPT = """
You are a SQL planning assistant for a teaching notebook.
Return exactly one JSON object and no Markdown.

Database: DuckDB with the spatial extension already installed and loaded.

Available tables:

1. osm_pois
Columns:
- osm_type VARCHAR
- osm_id VARCHAR
- name VARCHAR
- tag_key VARCHAR
- tag_value VARCHAR
- amenity VARCHAR
- tourism VARCHAR
- shop VARCHAR
- leisure VARCHAR
- public_transport VARCHAR
- railway VARCHAR
- highway VARCHAR
- historic VARCHAR
- office VARCHAR
- addr_street VARCHAR
- addr_housenumber VARCHAR
- lon VARCHAR, cast to DOUBLE for numeric use
- lat VARCHAR, cast to DOUBLE for numeric use
- all_tags_json VARCHAR

2. known_places
Columns:
- place_id VARCHAR: mainz_hbf, mainz_dom, mainz_theater
- label VARCHAR
- lat DOUBLE
- lon DOUBLE

Rules:
- Return JSON with: question, sql, map_title, center.
- center can be null or one of the known place IDs as a string.
- SQL must be read-only SELECT SQL. WITH ... SELECT is allowed.
- Query only osm_pois and optionally known_places.
- Include lon and lat in SELECT when the answer should be mapped.
- Use LIMIT 100 or smaller.
- Do not use INSERT, UPDATE, DELETE, DROP, CREATE, COPY, INSTALL, LOAD, ATTACH, DETACH, PRAGMA, or read_csv_auto.
- For distance or buffer-style queries, do not use degree distances or approximate formulas.
- For distance or buffer-style queries, create geometries with ST_Point(lon, lat), transform from EPSG:4326 to EPSG:25832 with ST_Transform(..., 'EPSG:4326', 'EPSG:25832', true), then use ST_DWithin and ST_Distance in meters.

Example JSON:
{
  "question": "Find cafes within 800 meters of Mainz Hauptbahnhof.",
  "center": "mainz_hbf",
  "map_title": "Cafes within 800 m of Mainz Hbf",
  "sql": "WITH center_projected AS (SELECT ST_Transform(ST_Point(lon, lat), 'EPSG:4326', 'EPSG:25832', true) AS geom_25832 FROM known_places WHERE place_id = 'mainz_hbf'), pois_projected AS (SELECT p.name, p.amenity, CAST(p.lon AS DOUBLE) AS lon, CAST(p.lat AS DOUBLE) AS lat, ST_Transform(ST_Point(CAST(p.lon AS DOUBLE), CAST(p.lat AS DOUBLE)), 'EPSG:4326', 'EPSG:25832', true) AS geom_25832 FROM osm_pois AS p WHERE p.amenity = 'cafe'), ranked AS (SELECT p.name, p.amenity, p.lon, p.lat, ST_Distance(p.geom_25832, c.geom_25832) AS distance_m FROM pois_projected AS p CROSS JOIN center_projected AS c WHERE ST_DWithin(p.geom_25832, c.geom_25832, 800)) SELECT name, amenity, lon, lat, round(distance_m, 1) AS distance_m FROM ranked ORDER BY distance_m LIMIT 50"
}
""".strip()


def response_text_from_chat_response(response: dict[str, Any]) -> str:
    choices = response.get("choices") or []
    if not choices:
        raise RuntimeError("The chat response did not contain any choices.")
    choice = choices[0]
    message = choice.get("message") or {}
    content = message.get("content")
    if isinstance(content, str) and content.strip():
        return content
    if isinstance(content, list):
        parts = []
        for item in content:
            if isinstance(item, dict):
                parts.append(str(item.get("text") or item.get("content") or ""))
            else:
                parts.append(str(item))
        text = "".join(parts).strip()
        if text:
            return text
    for key in ["reasoning_content", "reasoning", "text"]:
        value = message.get(key) or choice.get(key)
        if isinstance(value, str) and value.strip():
            return value
    preview_json(response, max_chars=2500)
    raise RuntimeError("The chat response did not contain text. Choose another listed chat_model.")


def extract_json_object(text: str | None) -> dict[str, Any]:
    if not isinstance(text, str) or not text.strip():
        raise ValueError("Expected non-empty text containing one JSON object.")
    text = text.strip()
    if text.startswith("```"):
        text = re.sub(r"^```(?:json)?", "", text).strip()
        text = re.sub(r"```$", "", text).strip()
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        match = re.search(r"\{.*\}", text, flags=re.DOTALL)
        if not match:
            print(text[:2000])
            raise ValueError("The model response did not contain a parseable JSON object.")
        return json.loads(match.group(0))


def call_sql_planner(question: str, model: str | None = None) -> dict[str, Any]:
    payload = {
        "model": model or chat_model,
        "messages": [
            {"role": "system", "content": SQL_PLANNER_SYSTEM_PROMPT},
            {"role": "user", "content": question},
        ],
        "temperature": 0.1,
        "max_tokens": 1800,
    }
    return ki_chat_request("POST", "/chat/completions", json=payload)


def parse_sql_planner_response(response: dict[str, Any]) -> tuple[str, dict[str, Any]]:
    raw_text = response_text_from_chat_response(response)
    plan = extract_json_object(raw_text)
    return raw_text, plan


## 9. Validate Model SQL

This validator is intentionally simple and conservative. It rejects obvious non-read-only SQL and makes sure the query references the fixture table.

In [ ]:
class SQLValidationError(ValueError):
    pass


FORBIDDEN_SQL_WORDS = {
    "insert", "update", "delete", "drop", "alter", "create", "copy", "install", "load",
    "attach", "detach", "pragma", "call", "export", "import", "read_csv", "read_csv_auto",
}


def normalize_sql_plan(plan: dict[str, Any]) -> dict[str, Any]:
    normalized = dict(plan)
    center = normalized.get("center")
    if isinstance(center, str) and center in KNOWN_PLACES:
        normalized["center_place"] = {"place_id": center, **KNOWN_PLACES[center]}
    elif center is None or center == "":
        normalized["center_place"] = None
    elif isinstance(center, dict):
        normalized["center_place"] = center
    else:
        raise SQLValidationError(f"Unknown center value: {center!r}")
    return normalized


def validate_sql(sql: str) -> str:
    if not isinstance(sql, str) or not sql.strip():
        raise SQLValidationError("SQL must be a non-empty string.")
    cleaned = sql.strip().rstrip(";").strip()
    lowered = cleaned.lower()

    if ";" in cleaned:
        raise SQLValidationError("Only one SQL statement is allowed.")
    if not (lowered.startswith("select") or lowered.startswith("with")):
        raise SQLValidationError("SQL must start with SELECT or WITH.")
    for word in FORBIDDEN_SQL_WORDS:
        if re.search(rf"\b{re.escape(word)}\b", lowered):
            raise SQLValidationError(f"Forbidden SQL keyword/function: {word}")
    if not re.search(r"\bosm_pois\b", lowered):
        raise SQLValidationError("SQL must query the osm_pois table.")
    if any(token in lowered for token in ["distance_m", "st_distance", "st_dwithin", "buffer"]):
        if "st_transform" not in lowered or "epsg:25832" not in lowered:
            raise SQLValidationError(
                "Distance/buffer SQL must use ST_Transform to EPSG:25832 before metric distance operations."
            )
    if any(token in lowered for token in ["111320", "haversine", "radians(", "cos("]):
        raise SQLValidationError("Use projected DuckDB spatial functions instead of approximate degree formulas.")
    if not re.search(r"\blimit\s+\d+\b", lowered):
        cleaned = f"SELECT * FROM ({cleaned}) AS model_query LIMIT 100"
    return cleaned


def validate_sql_plan(plan: dict[str, Any]) -> dict[str, Any]:
    normalized = normalize_sql_plan(plan)
    normalized["sql_model"] = normalized.get("sql")
    normalized["sql_final"] = validate_sql(normalized.get("sql", ""))
    if not normalized.get("map_title"):
        normalized["map_title"] = "Mainz OSM SQL result"
    return normalized


def show_model_sql(plan: dict[str, Any]) -> None:
    print("MODEL RETURNED SQL:")
    print(plan.get("sql_model") or "<model did not return sql>")
    print("\nSQL THAT WILL BE EXECUTED:")
    print(plan.get("sql_final"))


## 10. Ask the Model for SQL and Inspect the Result

This cell only calls the model and displays what it returned.

In [ ]:
question = "Find cafes within 800 meters of Mainz Hauptbahnhof."
planner_response = call_sql_planner(question)
raw_model_text, model_plan = parse_sql_planner_response(planner_response)

print("RAW MODEL TEXT:")
print(raw_model_text)
print("\nPARSED JSON PLAN:")
preview_json(model_plan)
print("\nMODEL RETURNED SQL:")
print(model_plan.get("sql") or "<model did not return sql>")


## 11. Validate, Execute, and Map the Model SQL

If validation fails, inspect the model SQL and edit it manually in the next section.

In [ ]:
validated_plan = validate_sql_plan(model_plan)
preview_json(validated_plan)
print()
show_model_sql(validated_plan)

model_df = con.execute(validated_plan["sql_final"]).fetchdf()
display(model_df)
map_sql_result(model_df, validated_plan["map_title"])


## 12. Manual SQL Repair Cell

Use this when the model returns SQL that is close but not executable. Copy the model SQL here, edit it, then run validation and execution again.

In [ ]:
sql_to_debug = """
-- Paste and edit model SQL here.
SELECT name, amenity, CAST(lon AS DOUBLE) AS lon, CAST(lat AS DOUBLE) AS lat
FROM osm_pois
WHERE amenity = 'cafe'
LIMIT 20
"""

sql_to_run = validate_sql(sql_to_debug)
print(sql_to_run)
debug_df = con.execute(sql_to_run).fetchdf()
display(debug_df)
map_sql_result(debug_df, "Manually repaired SQL result")


## 13. Test Prompts

Run the model section again with these prompts. Always inspect the SQL before executing it.

1. `Find cafes within 800 meters of Mainz Hauptbahnhof.`
2. `Show bicycle parking locations near Mainz Cathedral.`
3. `List museums or tourist attractions in Mainz city center.`
4. `Find supermarkets in the dataset and show them on a map.`
5. Guardrail test: `Delete all cafes from the table.` The validator should reject non-read-only SQL.

## 14. Student Tasks

1. Compare the hand-written SQL with the model-generated SQL.
2. Add a new known place to the `known_places` table.
3. Improve the SQL validator to reject queries without `lon` and `lat` when a map is requested.
4. Ask the model for a query that fails, then repair the SQL manually.
5. Explain why DuckDB over a local fixture is more stable than live Overpass API calls for this lesson.